In [ ]:
import base64, glob, hashlib, io, os, shutil, subprocess, tarfile
B64 = '...'
raw = base64.b64decode(B64)
assert hashlib.sha256(raw).hexdigest() == '01a5a6a629da34a0b19a114e66de333da1d4e02f6cc038593e6bf9881f0bfee1'
REPO, OUT = '/kaggle/working/repo', '/kaggle/working/paper'
tarfile.open(fileobj=io.BytesIO(raw), mode='r:*').extractall(REPO)
print('paper sources sha256 01a5a6a629da34a0b19a114e66de333da1d4e02f6cc038593e6bf9881f0bfee1 from git 461127e+local changes')
P = REPO + '/paper/puformer'


def sh(cmd, cwd=P):
    r = subprocess.run(cmd, shell=True, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    print('\n'.join(l for l in r.stdout.splitlines() if 'Underfull' not in l)[-6000:], flush=True)
    assert r.returncode == 0, cmd


# .npz files are not in git: take them from the analysis kernels' outputs
for src in glob.glob('/kaggle/input/**/visual.npz', recursive=True) + glob.glob('/kaggle/input/**/sam_maps.npz', recursive=True):
    ds = 'puformer_pavia_x4' if 'pavia' in src.lower() else 'puformer_chikusei_x4_v3'
    sub = 'sam_analysis' if src.endswith('sam_maps.npz') else 'paper_analysis'
    dst = f'{REPO}/results/{ds}/{sub}/' + os.path.basename(src)
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    shutil.copyfile(src, dst)
    print('npz:', src, '->', dst)
if True:
    sh('python make_numbers.py && python make_tables.py && python make_figures.py && python make_q1_tables.py && python novelty_check.py')
    preds = sorted(glob.glob('/kaggle/input/**/bench-*/test_pred_f16.npy', recursive=True))
    pdir = os.path.dirname(os.path.dirname(preds[0])) if preds else ''
    sh(f'Q1_PREDS={pdir} python make_q1_figures.py')
os.makedirs('/kaggle/working/bin', exist_ok=True)
# the statically linked (musl) build: the glibc build needs a newer libc than Kaggle's image has
for v in ['0.15.0', '0.14.1']:
    url = (f'https://github.com/tectonic-typesetting/tectonic/releases/download/tectonic%40{v}/'
           f'tectonic-{v}-x86_64-unknown-linux-musl.tar.gz')
    if subprocess.run(f'curl -fsSL {url} | tar xz', shell=True, cwd='/kaggle/working/bin').returncode == 0:
        break
sh('./tectonic --version', cwd='/kaggle/working/bin')
sh('/kaggle/working/bin/tectonic -X compile main.tex')
sh('/kaggle/working/bin/tectonic -X compile supplementary.tex')
sh('/kaggle/working/bin/tectonic -X compile --keep-intermediates cvpr_main.tex')
sh('python cvpr_refs.py')
sh('/kaggle/working/bin/tectonic -X compile cvpr_supp.tex')
os.makedirs(OUT, exist_ok=True)
for f in ['main.pdf', 'supplementary.pdf', 'cvpr_main.pdf', 'cvpr_supp.pdf', 'numbers.tex']:
    shutil.copyfile(f'{P}/{f}', f'{OUT}/{f}')
for d in ['figures', 'tables']:
    if os.path.isdir(f'{P}/{d}'):
        shutil.copytree(f'{P}/{d}', f'{OUT}/{d}', dirs_exist_ok=True)
# page previews and the extracted text, for checking the layout and unresolved references (??)
sh('pip install -q pymupdf')
import fitz
os.makedirs(f'{OUT}/pages', exist_ok=True)
for f in ['main', 'supplementary', 'cvpr_main', 'cvpr_supp']:
    with fitz.open(f'{OUT}/{f}.pdf') as doc:
        text = '\n'.join(p.get_text() for p in doc)
        open(f'{OUT}/pages/{f}.txt', 'w').write(text)
        print(f, len(doc), 'pages; unresolved references:', text.count('??'))
        for i, p in enumerate(doc):
            p.get_pixmap(dpi=110).save(f'{OUT}/pages/{f}-{i + 1:02d}.png')
sh('ls -laR ' + OUT, cwd=OUT)
